In [1]:
import os

os.chdir(os.path.expanduser("~/classifier/src"))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from warnings import filterwarnings
from types import SimpleNamespace as sn
from yaml import load, FullLoader
from psycopg2 import connect

In [ ]:
class Filter(object):
	def __init__(self, df: pd.DataFrame, ratio: int = 10) -> None:
		self.maps = df
		self.dirty_maps, self.weird_maps = None, None
		self.filtered_maps = None
		self.filter_df(ratio) # автоматический вызов функции фильтрации

	def find_weird_maps(self, px: int = 3) -> list:
		'''Не совпадение центра карты с максимум интенсивности'''
		df = self.maps
		weird_maps = [
			file for c_x, x, c_y, y, a, file in
			zip(df.mapc_x, df.map_max_x, df.mapc_y, 
	   		df.map_max_y, df.obs_author, df.file_name)
			if (abs(c_x - x) > px or abs(c_y - y) > px) and a != 'Alan Marscher'
		]
		return weird_maps
	
	def maps_w_bad_signal_noise(self, ratio: int) -> dict:
		'''Маленький сигнал/шуму'''
		df = self.maps
		signal_noise = {
			file: sig / nl for file, sig, nl in 
			zip(df.file_name, df.map_max, df.noise_level)
			if sig/nl <= ratio
		}
		return signal_noise
	
	def filter_df(self, ratio: int) -> None:
		'''Сигнал/шум + координаты центра'''
		self.weird_maps = set(self.find_weird_maps())
		self.dirty_maps = self.maps_w_bad_signal_noise(ratio)
		self.filtered_maps = self.weird_maps.union(self.dirty_maps)

		for x in self.maps.index:
			b_maj, b_min = self.maps.loc[x, 'b_maj'], self.maps.loc[x, 'b_min']
			file_name = self.maps.loc[x, 'file_name']
			pixel_size = self.maps.loc[x, 'pixel_size_y']
			if (b_maj == -1 or b_min == -1 or 
	   			file_name in self.filtered_maps or
				b_maj * 3.6e6 / pixel_size > 60):
				self.maps.drop(x, inplace=True)

In [ ]:
class BeamCluster(Filter):
	test_dir = 'src/astrogeo/test'
	def __init__(self, df: pd.DataFrame, ratio: int = 10) -> None:
		Filter.__init__(self, df, ratio) # <--- фильтрация базы данных
		self.kmeans, self.X = None, None
		self.b_maj, self.b_min = None, None
		if not os.path.exists(self.test_dir):
			os.makedirs(self.test_dir)
	
	def _preprocess(self) -> np.array:
		'''Предобработка'''
		pixel_size = self.maps['pixel_size_y'].to_numpy().T
		self.b_maj = self.maps['b_maj'].to_numpy().T * 3.6e6 / pixel_size
		self.b_min = self.maps['b_min'].to_numpy().T * 3.6e6 / pixel_size
		self.b_pa = self.maps['b_pa'].to_numpy().T

		return np.stack([self.b_maj, self.b_min, self.b_pa])
	
	
	def _beam_clustering(self, clusters: int) -> None:
		'''Кластеризация'''
		X = self._preprocess()
		kms = KMeans(n_clusters=clusters, random_state=0, n_init='auto')
		self.kmeans = kms.fit(X.T)
		labels = np.array([self.kmeans.labels_])
		self.X = np.concatenate((X.T, labels.T), axis=1)

	
	def beam_cluster_means(self, clusters: int) -> pd.DataFrame:
		'''Метод для получения итоговых параметров кластеров'''
		if self.X is None:
			self._beam_clustering(clusters)
		
		data = self.X
		means = {}
		for ind, label in enumerate(data[:, 3]):
			label = int(label)
			if label not in means:
				means[label] = [data[ind, 0], data[ind, 1], data[ind, 2], 1]
			else:
				for i in range(3):
					means[label][i] += data[ind, i]
				means[label][3] += 1
		
		for label in means:
			count = means[label][3]
			for i in range(3):
				means[label][i] /= count
		
		df = pd.DataFrame(means).T
		df = df.rename(
			columns={0: 'b_maj', 1: 'b_min', 2: 'b_pa', 3: 'amount'}
		)
		df.index.name = 'Cluster'
		df.amount = df.amount.astype(int)
		for col in df.columns:
			if col != 'amount':
				df[col] = df[col].apply(
					lambda x: np.format_float_scientific(x, precision=2)
				)
		df = df.astype(float)
		return df.sort_index()

# Подключение к базе данных

In [7]:
filterwarnings('ignore')
with open('../config/config.yaml') as f:
    config = sn(**load(f, Loader=FullLoader))
config_db, path = sn(**config.db), config.fits_path

cnx = connect(
    host=config_db.host, dbname=config_db.dbname,
    user=config_db.user, password=config_db.psswd
)
maps = pd.read_sql('select * from maps24;', con=cnx)

In [ ]:
ratio = 10
b = BeamCluster(maps, ratio)
# df = b.beam_cluster_means(ratio) # <--- получение кластеров